# Amazon Inventory Reconciliation Using AI
Aarush Patil - PES1UG24AM005. Run on a Colab **T4 GPU**, top to bottom.

In [1]:
import torch
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - switch runtime to T4')

Tesla T4


In [2]:
import os, zipfile
if not os.path.exists('/content/amazon-inventory-reconciliation'):
    from google.colab import files
    up = files.upload()   # the repo zip
    zipfile.ZipFile(list(up)[0]).extractall('/content')
%cd /content/amazon-inventory-reconciliation
!pip -q install scikit-image tqdm

Saving amazon-inventory-reconciliation-final (1).zip to amazon-inventory-reconciliation-final (1).zip
/content/amazon-inventory-reconciliation


## Data

In [3]:
!python download_data.py --n 30000 --out data --workers 32
!python build_splits.py --data data

images kept: 100% 30000/30000 [04:01<00:00, 124.43it/s]
done: 30000 images -> data/images
class counts (items: images): {0: 848, 1: 3368, 2: 6387, 3: 7598, 4: 6660, 5: 5139}
{'n_images': 30000, 'split_sizes': {'train': 21000, 'val': 6000, 'test': 3000}, 'class_counts': {'0': 848, '1': 3368, '2': 6387, '3': 7598, '4': 6660, '5': 5139}, 'majority_class_share': 0.25326666666666664}


## Baselines

In [4]:
!python baselines.py --data data --out results/baselines.json

loading raw pixels ...
PCA(256) keeps 93.4% variance
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
logreg_pca                         val acc 0.295 | test acc 0.285 rmse 1.37
linear_svc_pca                     val acc 0.294 | test acc 0.277 rmse 1.37
decision_tree_pca                  val acc 0.249 | test acc 0.255 rmse 1.57
rbf_svm_pca                        val acc 0.302 | test acc 0.298 rmse 1.32
computing HOG ...
rbf_svm_hog                        val acc 0.307 | test acc 0.319 rmse 1.32
best baseline by val acc: rbf_svm_hog 

## ResNet34: softmax head and ordinal head

In [5]:
!python train_cnn.py --data data --arch resnet34 --head cls --name r34_cls --lr-find --workers 2

run r34_cls | device cuda | amp True
Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth
100% 83.3M/83.3M [00:00<00:00, 156MB/s]
[head] ep  1 | loss 1.627 train acc 0.255 | val acc 0.294 rmse 1.522
[head] ep  2 | loss 1.530 train acc 0.295 | val acc 0.312 rmse 1.307
LR finder: suggested 2.92e-05 -> using head lr 2.92e-05
[full] ep  1 | loss 1.430 train acc 0.336 | val acc 0.372 rmse 1.175
[full] ep  2 | loss 1.327 train acc 0.390 | val acc 0.389 rmse 1.153
[full] ep  3 | loss 1.289 train acc 0.413 | val acc 0.391 rmse 1.142
[full] ep  4 | loss 1.273 train acc 0.420 | val acc 0.406 rmse 1.102
[full] ep  5 | loss 1.206 train acc 0.459 | val acc 0.413 rmse 1.096
[full] ep  6 | loss 1.147 train acc 0.491 | val acc 0.417 rmse 1.096
[full] ep  7 | loss 1.107 train acc 0.517 | val acc 0.417 rmse 1.105
[full] ep  8 | loss 1.071 train acc 0.542 | val acc 0.417 rmse 1.104
[full] ep  9 | loss 1.061 train acc 0.550 | 

In [6]:
!python train_cnn.py --data data --arch resnet34 --head ordinal --name r34_ord --lr-find --workers 2

run r34_ord | device cuda | amp True
[head] ep  1 | loss 0.434 train acc 0.272 | val acc 0.297 rmse 1.195
[head] ep  2 | loss 0.405 train acc 0.287 | val acc 0.302 rmse 1.183
LR finder: suggested 2.54e-06 -> using head lr 1.00e-05
[full] ep  1 | loss 0.380 train acc 0.320 | val acc 0.350 rmse 1.099
[full] ep  2 | loss 0.356 train acc 0.352 | val acc 0.361 rmse 1.086
[full] ep  3 | loss 0.350 train acc 0.356 | val acc 0.361 rmse 1.079
[full] ep  4 | loss 0.343 train acc 0.364 | val acc 0.366 rmse 1.055
[full] ep  5 | loss 0.330 train acc 0.385 | val acc 0.381 rmse 1.045
[full] ep  6 | loss 0.320 train acc 0.395 | val acc 0.386 rmse 1.038
[full] ep  7 | loss 0.314 train acc 0.405 | val acc 0.390 rmse 1.035
[full] ep  8 | loss 0.310 train acc 0.410 | val acc 0.390 rmse 1.030
[full] ep  9 | loss 0.310 train acc 0.409 | val acc 0.391 rmse 1.027
val       acc 0.3913 rmse 1.027 mae 0.749 within1 0.873
val_tta   acc 0.3932 rmse 1.012 mae 0.739 within1 0.879
test      acc 0.3887 rmse 1.034 mae 

## Results

In [7]:
!python make_results.py --data data --baselines results/baselines.json --out results
!cat results/results.json | head -c 2500

{
  "cnn_name": "r34_cls",
  "cnn_acc": 0.4116666666666667,
  "cnn_rmse": 1.1073993558483468,
  "cnn_mae": 0.7816666666666666,
  "cnn_within1": 0.8336666666666667,
  "cnn_acc_no_tta": 0.4066666666666667,
  "x_over_random": 2.47,
  "baseline_name": "rbf_svm_hog",
  "baseline_acc": 0.31933333333333336,
  "baseline_rmse": 1.3178264933847197,
  "rel_improvement": 0.28914405010438404
}
wrote results/results.json and figures
{
  "generated": "2026-10-15",
  "paper_ref": {
    "cnn_resnet34_adam_324k": {
      "acc": 0.562,
      "rmse": 0.9
    },
    "cnn_resnet34_sgdr_150k": {
      "acc": 0.538,
      "rmse": 0.94
    },
    "svm_raw": {
      "acc": 0.26
    },
    "svm_pca": {
      "acc": 0.32
    }
  },
  "data": {
    "n_images": 30000,
    "split_sizes": {
      "train": 21000,
      "val": 6000,
      "test": 3000
    },
    "class_counts": {
      "0": 848,
      "1": 3368,
      "2": 6387,
      "3": 7598,
      "4": 6660,
      "5": 5139
    },
    "majority_class_share": 0.2532

In [8]:
!pip -q install gradio
!python demo.py --run runs/r34_cls --gradio

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://4fb1076c675425a9be.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Keyboard interruption in main thread... closing server.
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 3957, in block_thread
    time.sleep(0.1)
    ~~~~~~~~~~^^^^^
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/content/amazon-inventory-reconciliation/demo.py", line 93, in <module>
    main()
    ~~~~^^
  File "/content/amazon-inventory-reconciliation/demo.py", line 89, in main
    flagging_mode="never").launch(share=True)
                           ~~~~~~^^^^^^^^^^^^
  File "/usr/local/lib